# Gravitino + Daft + S3/MinIO 探索 Notebook

本 notebook 演示在本地 Gravitino Playground 中，把 MinIO 作为 S3 后端，通过 Gravitino 管理 fileset 元数据，并用 Daft 读写底层 S3 对象。

前置条件：
- `gravitino-playground` 已启动，且 MinIO 暴露在 http://localhost:9000。
- 已运行 `scripts/bootstrap_s3.py` 创建 S3-backed fileset catalog/schema/filesets。
- 已运行 `scripts/upload_s3_fileset.py` 上传示例数据。
- 宿主机能把 `minio` 解析到 `127.0.0.1`，例如执行过：
  ```bash
  sudo sh -c 'echo "127.0.0.1 minio" >> /etc/hosts'
  ```

In [ ]:
import sys
sys.path.insert(0, '..')

from gravitino_daft.config import (
    GRAVITINO_ENDPOINT, GRAVITINO_METALAKE,
    S3_CATALOG, S3_SCHEMA, S3_INPUT_FILESET, S3_OUTPUT_FILESET,
    gravitino_io_config, s3_io_config, s3_gvfs_path, s3_actual_path, headers
)
import daft
import requests

## 1. 通过 Gravitino REST API 查看 fileset 元数据

In [ ]:
url = (
    f'{GRAVITINO_ENDPOINT}/api/metalakes/{GRAVITINO_METALAKE}/'
    f'catalogs/{S3_CATALOG}/schemas/{S3_SCHEMA}/filesets/{S3_INPUT_FILESET}'
)
resp = requests.get(url, headers=headers())
resp.raise_for_status()
fileset = resp.json()['fileset']
storage_location = fileset['storageLocation']
print('Fileset name:', fileset['name'])
print('Storage location (Gravitino):', storage_location)
print('Daft S3 path:', storage_location.replace('s3a://', 's3://'))

## 2. 用 Daft + GVFS 读取 S3-backed fileset

这是最直接的方式：Daft 的 `GravitinoConfig` 会自动把 ``gvfs://fileset/...`` 解析到 S3 后端。

In [ ]:
io_config = gravitino_io_config()
parquet_uri = s3_gvfs_path(S3_INPUT_FILESET, '*.parquet')
df = daft.read_parquet(parquet_uri, io_config=io_config)
df.show()

## 3. 用 Daft + GVFS 向 S3-backed fileset 写入数据并读回

In [ ]:
sample = daft.from_pydict({
    'id': [1, 2, 3],
    'label': ['x', 'y', 'z'],
    'value': [1.1, 2.2, 3.3],
})
output_uri = s3_gvfs_path(S3_OUTPUT_FILESET, 'notebook_output.parquet')
sample.write_parquet(output_uri, io_config=io_config)

read_back = daft.read_parquet(output_uri, io_config=io_config)
read_back.show()

## 4. 列出 fileset 下的所有对象

In [ ]:
glob_uri = s3_gvfs_path(S3_INPUT_FILESET, '**/*')
files_df = daft.from_glob_path(glob_uri, io_config=io_config)
files_df.show()

## 5. （可选）用 Daft 直连底层 S3

如果你希望绕过 GVFS、自己管理 S3 凭证和端点，可以先用 Gravitino 拿到物理位置，再用 `S3Config` 直连 MinIO。

In [ ]:
s3_config = s3_io_config()
direct_uri = s3_actual_path(S3_INPUT_FILESET, '*.parquet')
direct_df = daft.read_parquet(direct_uri, io_config=s3_config)
direct_df.show()